# 05. Speculative Decoding：Draft、Verify、Accept

> 面向即将加入 SGLang 开源社区的开发者。建议从仓库根目录启动 Jupyter，
> 或者在 notebook 第一格运行路径检查。本文以本地源码为主，线上文档为索引。
> Markdown 中的源码摘录会插入 `# 注：...` 行内讲解；可执行代码 cell 则保持可运行。


In [ ]:
from pathlib import Path
import ast, inspect, re, textwrap

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "python" / "sglang").exists() and (candidate / "docs").exists():
            return candidate
    raise RuntimeError("没有找到 SGLang 仓库根目录，请从仓库内启动 notebook。")

ROOT = find_repo_root()
print(ROOT)

def read_rel(path):
    return (ROOT / path).read_text()

def show_lines(path, start, end):
    lines = read_rel(path).splitlines()
    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:4d}: {lines[i-1]}")

def find_defs(path, names=None):
    tree = ast.parse(read_rel(path))
    rows = []
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            if names is None or node.name in names:
                rows.append((node.lineno, type(node).__name__, node.name))
    return sorted(rows)


## 统一心智模型

Speculative decoding 的共同目标是减少 target model 的 decode 次数：

1. draft 产生多个候选 token。
2. target 一次 verify 这些候选。
3. accept/reject 决定提交几个 token。
4. KV cache、seq len、输出 token、采样状态一起前进。

不同算法的差异在 draft 来源：

- `EAGLE` / `EAGLE3`：用 draft model / hidden states 预测树状候选。
- `STANDALONE`：小 draft model 独立生成候选。
- `NGRAM`：从历史 ngram 中检索候选，不需要 draft KV。
- `DFLASH` / `FROZEN_KV_MTP`：更新的内部路径，围绕 target hidden / frozen KV / block verify 做优化。


In [ ]:
show_lines("python/sglang/srt/speculative/spec_info.py", 20, 120)


### `SpeculativeAlgorithm`：算法能力比 enum 名字更重要

```python
# python/sglang/srt/speculative/spec_info.py:20-120
  20:     from sglang.srt.managers.overlap_utils import FutureMap
  21:     from sglang.srt.managers.schedule_batch import ScheduleBatch
  22:     from sglang.srt.managers.tp_worker import TpModelWorker
  23:     from sglang.srt.server_args import ServerArgs
  24:     from sglang.srt.speculative.base_spec_worker import BaseSpecWorker
  25:     from sglang.srt.speculative.ngram_worker import NGRAMWorker
  26: 
  27: 
  28: class SpeculativeAlgorithm(Enum):
  29:     """Builtin speculative decoding algorithms. Plugin-registered ones are
  30:     ``CustomSpecAlgo`` instances; ``from_string`` returns either type, and
  31:     both expose the same ``is_*()`` / ``create_worker`` interface so callers
  32:     dispatch uniformly without isinstance checks.
  33:     """
  34: 
  35:     DFLASH = auto()
  36:     EAGLE = auto()
  37:     EAGLE3 = auto()
  38:     FROZEN_KV_MTP = auto()
  39:     STANDALONE = auto()
  40:     NGRAM = auto()
  41:     NONE = auto()
  42: 
  43:     @classmethod
  44:     def from_string(
  45:         cls, name: Optional[str]
  46:     ) -> Union[SpeculativeAlgorithm, CustomSpecAlgo]:
  47:         if name is None:
  48:             return cls.NONE
  49:         upper = name.upper()
  50:         try:
  51:             return cls[upper]
  52:         except KeyError:
  53:             pass
  54:         spec = _get_registered_spec(upper)
      # 注：`spec` 接收 `_get_registered_spec` 的结果：查询插件注册的 speculative algorithm。
  55:         if spec is not None:
  56:             return spec
  57:         raise ValueError(f"Unknown speculative algorithm name: {name}")
  58: 
  59:     @classmethod
  60:     def register(
  61:         cls,
  62:         name: str,
  63:         *,
  64:         supports_overlap: bool = False,
  65:         validate_server_args: Optional[ServerArgsValidator] = None,
  66:         spec_class: Type[CustomSpecAlgo] = CustomSpecAlgo,
  67:     ) -> Callable[[WorkerFactory], WorkerFactory]:
  68:         """Decorator to register a plugin speculative algorithm. The factory
  69:         takes ``server_args`` and returns the worker class. Pass a
  70:         ``CustomSpecAlgo`` subclass via ``spec_class`` to override any
  71:         ``is_*()`` / ``create_worker`` method.
  72: 
  73:         Example:
  74:             @SpeculativeAlgorithm.register("MY_SPEC", supports_overlap=False)
  75:             def _factory(server_args):
  76:                 return MySpecWorker
  77:         """
  78:         return _register_algorithm(
  79:             name,
  80:             supports_overlap=supports_overlap,
  81:             validate_server_args=validate_server_args,
  82:             spec_class=spec_class,
  83:         )
  84: 
  85:     def is_some(self) -> bool:
  86:         return self != SpeculativeAlgorithm.NONE
  87: 
  88:     def is_none(self) -> bool:
  89:         return self == SpeculativeAlgorithm.NONE
  90: 
  91:     def is_speculative(self) -> bool:
  92:         return self != SpeculativeAlgorithm.NONE
  93: 
  94:     def is_eagle(self) -> bool:
  95:         # FIXME(kpham_sgl): Remove FROZEN_KV_MTP here once we
  96:         # have established support for it in the scheduler.
  97:         return self in (
  98:             SpeculativeAlgorithm.EAGLE,
  99:             SpeculativeAlgorithm.EAGLE3,
 100:             SpeculativeAlgorithm.FROZEN_KV_MTP,
 101:         )
 102: 
 103:     def is_eagle3(self) -> bool:
 104:         return self == SpeculativeAlgorithm.EAGLE3
 105: 
 106:     def is_frozen_kv_mtp(self) -> bool:
 107:         return self == SpeculativeAlgorithm.FROZEN_KV_MTP
 108: 
 109:     def is_dflash(self) -> bool:
 110:         return self == SpeculativeAlgorithm.DFLASH
 111: 
 112:     def is_standalone(self) -> bool:
 113:         return self == SpeculativeAlgorithm.STANDALONE
 114: 
 115:     def is_ngram(self) -> bool:
 116:         return self == SpeculativeAlgorithm.NGRAM
 117: 
 118:     def supports_target_verify_for_draft(self) -> bool:
 119:         return self.is_dflash()
 120: 
```

**读这段时抓住：**

- `from_string` 同时解析内建算法和插件注册算法，所以调用方不能假设返回值一定是 Enum。
- `is_eagle` 把 EAGLE、EAGLE3、FROZEN_KV_MTP 归入同一族，是因为 scheduler 视角上它们共享 draft hidden/KV 约束。
- `has_draft_kv` 区分 NGRAM 这种不写 draft KV 的算法，直接影响每步 KV reserve 估算。
- `need_topk` 决定 target/draft forward 是否需要额外 top-k 信息。


## `SpeculativeAlgorithm` 是调度分发的统一接口

注意这里不是简单 enum。插件也可以注册 custom speculative algorithm，但必须 duck-type 出相同的 `is_*` / `supports_*` 接口。这样 scheduler/model runner 可以统一调用。


In [ ]:
show_lines("python/sglang/srt/speculative/spec_registry.py", 24, 130)
show_lines("python/sglang/srt/speculative/spec_info.py", 150, 225)


### `CustomSpecAlgo`：插件算法必须 duck-type 内建接口

```python
# python/sglang/srt/speculative/spec_registry.py:24-130
  24: class CustomSpecAlgo:
  25:     """A plugin-registered speculative algorithm. Duck-types
  26:     ``SpeculativeAlgorithm`` enum values (same ``is_*()`` / ``create_worker``
  27:     interface).
  28: 
  29:     Plugins may subclass this to override any ``is_*()`` / ``supports_*()`` /
  30:     ``create_worker`` method (e.g. to integrate with builtin-specific
  31:     branches like ``if spec_algorithm.is_eagle():`` in scheduler /
  32:     model_runner). Pass the subclass via ``spec_class=...`` at registration.
  33: 
  34:     Defaults: all ``is_*()`` return ``False`` except ``is_speculative``.
  35: 
  36:     ``supports_overlap=False`` is deprecated: the spec V1 worker path has been
  37:     removed, so such algorithms run on the V2 scheduler schema with overlap
  38:     disabled (synchronous). Migrate plugin workers to the V2 schema and
  39:     overlap scheduling.
  40:     """
  41: 
  42:     def __init__(
  43:         self,
  44:         name: str,
  45:         factory: WorkerFactory,
  46:         *,
  47:         supports_overlap: bool = False,
  48:         validate_server_args: Optional[ServerArgsValidator] = None,
  49:     ):
  50:         self.name = name
  51:         self.factory = factory
  52:         self.supports_overlap = supports_overlap
  53:         self.validate_server_args = validate_server_args
  54: 
  55:     def __repr__(self) -> str:
  56:         return f"CustomSpecAlgo({self.name!r})"
  57: 
  58:     def is_some(self) -> bool:
  59:         return True
  60: 
  61:     def is_none(self) -> bool:
  62:         return False
  63: 
  64:     def is_speculative(self) -> bool:
  65:         return True
  66: 
  67:     def is_eagle(self) -> bool:
  68:         return False
  69: 
  70:     def is_eagle3(self) -> bool:
  71:         return False
  72: 
  73:     def is_frozen_kv_mtp(self) -> bool:
  74:         return False
  75: 
  76:     def is_dflash(self) -> bool:
  77:         return False
  78: 
  79:     def is_standalone(self) -> bool:
  80:         return False
  81: 
  82:     def is_ngram(self) -> bool:
  83:         return False
  84: 
  85:     def supports_target_verify_for_draft(self) -> bool:
  86:         return False
  87: 
  88:     def has_draft_kv(self) -> bool:
  89:         # Conservative default: the larger KV reserve.
  90:         return True
  91: 
  92:     def handle_server_args(self, server_args: ServerArgs) -> None:
  93:         pass
  94: 
  95:     def create_worker(self, server_args: ServerArgs) -> Type:
      # 注：`create_worker` 是插件 spec algorithm 进入 runtime 的收口点；最终必须返回 scheduler/model runner 可调用的 worker 类。
  96:         if not server_args.disable_overlap_schedule and not self.supports_overlap:
  97:             raise ValueError(
  98:                 f"Speculative algorithm {self.name} does not support overlap scheduling."
  99:             )
 100:         if not self.supports_overlap:
 101:             # Reached only when overlap is disabled, so the algorithm really
 102:             # does run synchronously on the V2 schema below.
 103:             logger.warning(
 104:                 "Speculative algorithm %s is registered with "
 105:                 "supports_overlap=False, which is deprecated: the spec V1 "
 106:                 "worker path has been removed, and the algorithm now runs on "
 107:                 "the V2 scheduler schema with overlap disabled (synchronous). "
 108:                 "Migrate the plugin worker to support overlap scheduling.",
 109:                 self.name,
 110:             )
 111:         return self.factory(server_args)
      # 注：关键调用：`self.factory` - 调用插件提供的 worker factory，得到自定义 speculative worker 类。
 112: 
 113:     def get_num_tokens_per_bs_for_target_verify(
 114:         self, num_draft_tokens: int, is_draft_worker: bool
 115:     ) -> int:
 116:         # FIXME: Remove this after the forward mode refactor. Target verify is
 117:         # essentially a fixed sequence length prefill/extend with full cuda
 118:         # graph support. We can use it for target verify, or we can use it for
 119:         # other cases which is not target verify but fixed length prefill.
 120:         # Here, we expose this interface to allow the other use cases.
 121:         return num_draft_tokens
 122: 
 123:     def build_disagg_draft_input(
      # 注：disaggregation 模式下插件可以构造 draft 阶段输入；默认 None 表示不参与 EPD/PD draft 输入改写。
 124:         self,
 125:         batch: ScheduleBatch,
 126:         server_args: ServerArgs,
 127:         last_tokens_tensor: torch.Tensor,
 128:         future_map: FutureMap,
 129:     ) -> Optional[SpecInput]:
 130:         return None
```

**读这段时抓住：**

- 自定义算法不是随便注册一个名字；它要暴露和 `SpeculativeAlgorithm` 相同的 `is_*` / `supports_*` 方法。
- `supports_overlap=False` 会影响 overlap schedule，可见 spec worker 和 scheduler 并不是松耦合插件。
- 如果插件算法要复用 EAGLE 分支，通常要提供自定义 `spec_class` 覆盖对应谓词。


### `create_worker`：算法名最后落到 worker 类

```python
# python/sglang/srt/speculative/spec_info.py:160-224
 160:         return self.is_eagle() or self.is_standalone()
 161: 
 162:     def handle_server_args(self, server_args: ServerArgs) -> None:
 163:         """Hook for per-algorithm server args mutation.
 164: 
 165:         In-place updated.
 166:         """
 167:         from sglang.srt.arg_groups.speculative_hook import (
 168:             _handle_dflash,
 169:             _handle_eagle_family,
 170:             _handle_frozen_kv_mtp,
 171:             _handle_ngram,
 172:         )
 173: 
 174:         if self.is_dflash():
 175:             _handle_dflash(server_args)
 176:         elif self.is_frozen_kv_mtp():
 177:             _handle_frozen_kv_mtp(server_args)
 178:         elif self.is_eagle() or self.is_standalone():
 179:             _handle_eagle_family(server_args)
 180:         elif self.is_ngram():
 181:             _handle_ngram(server_args)
 182: 
 183:     def get_num_tokens_per_bs_for_target_verify(
 184:         self, num_draft_tokens: int, is_draft_worker: bool
 185:     ) -> int:
 186:         # FIXME: Remove this after the forward mode refactor. Target verify is
 187:         # essentially a fixed sequence length prefill/extend with full cuda
 188:         # graph support. We can use it for target verify, or we can use it for
 189:         # other cases which is not target verify but fixed length prefill.
 190:         # Here, we expose this interface to allow the other use cases.
 191:         return num_draft_tokens
 192: 
 193:     def create_worker(
 194:         self, server_args: ServerArgs
 195:     ) -> Optional[Union[Type[BaseSpecWorker], Type[TpModelWorker], Type[NGRAMWorker]]]:
 196:         assert (
 197:             not self.is_none()
 198:         ), "Cannot create worker for NONE speculative algorithm."
 199: 
 200:         if self.is_dflash():
 201:             # V2 worker drives both overlap and non-overlap (scheduler runs it
 202:             # synchronously when overlap is disabled), same as EAGLE.
 203:             from sglang.srt.speculative.dflash_worker_v2 import DFlashWorkerV2
 204: 
 205:             return DFlashWorkerV2
 206: 
 207:         if self.is_frozen_kv_mtp():
 208:             # V2 worker drives both overlap and non-overlap (scheduler runs it
 209:             # synchronously when overlap is disabled), same as EAGLE.
 210:             from sglang.srt.speculative.frozen_kv_mtp_worker_v2 import (
 211:                 FrozenKVMTPWorkerV2,
 212:             )
 213: 
 214:             return FrozenKVMTPWorkerV2
 215: 
 216:         # EAGLE / EAGLE3 / STANDALONE / MULTI_LAYER always use the V2 worker,
 217:         # even with overlap disabled (scheduler drives it synchronously).
 218:         if self.is_eagle() and server_args.enable_multi_layer_eagle:
 219:             from sglang.srt.speculative.multi_layer_eagle_worker_v2 import (
 220:                 MultiLayerEagleWorkerV2,
 221:             )
 222: 
 223:             return MultiLayerEagleWorkerV2
 224: 
```

**读这段时抓住：**

- DFLASH、FROZEN_KV_MTP、EAGLE、STANDALONE、NGRAM 都在这里映射到不同 worker。
- EAGLE-family 默认使用 V2 worker，说明旧 spec worker 路径已经不是主线。
- 新增内建算法时，除了 server args hook，还要在这里定义 worker 映射。


## Worker 分层：先看 contract，再看真正闭环

`BaseSpecWorker` 只定义 target/draft worker 的共同 contract；真正的 speculative decode 闭环在具体 worker 的 `forward_batch_generation` 中。

读这部分时优先追三件事：

- 谁产生 draft 候选：EAGLE 是 draft model，NGRAM 是 CPU corpus。
- 谁执行 verify：最终都回到 target worker。
- verify 后如何返回 `GenerationBatchResult`：scheduler 后面只看 accept_lens、new_seq_lens、next_draft_input 等统一字段。


In [ ]:
for path in [
    "python/sglang/srt/speculative/base_spec_worker.py",
    "python/sglang/srt/speculative/eagle_worker_v2.py",
    "python/sglang/srt/speculative/ngram_worker.py",
    "python/sglang/srt/speculative/reject_sampling.py",
]:
    print("\n==", path)
    for lineno, kind, name in find_defs(path):
        if name in {
            "BaseSpecWorker", "EagleDraftWorkerBase", "EAGLEWorkerV2", "EagleDraftWorker",
            "NGRAMWorker", "draft", "draft_forward", "verify", "forward_batch_generation",
            "_prepare_for_speculative_decoding", "_update_ngram_corpus",
        }:
            print(f"{lineno:4d} {kind:16s} {name}")


### `BaseSpecWorker`：target worker 和 draft worker 的共同 contract

```python
# python/sglang/srt/speculative/base_spec_worker.py:274-329
 274: class BaseSpecWorker(ABC):
 275:     @property
 276:     @abstractmethod
 277:     def target_worker(self) -> TpModelWorker:
 278:         pass
 279: 
 280:     @property
 281:     @abstractmethod
 282:     def draft_worker(self) -> EagleDraftWorkerBase:
 283:         pass
 284: 
 285:     @property
      # 注：`war_fastpath_runner` 指向最后读取共享 buffer 的 runner，overlap scheduler 的 WAR barrier 会等它的 read-done event。
 286:     def war_fastpath_runner(self):
 287:         # The runner that runs the step's LAST shared-buffer-reading phase --
 288:         # it owns the read-done event the scheduler's WAR barrier waits on.
 289:         # Default is the target runner; override if the last phase runs
 290:         # elsewhere (eagle's draft_extend runs on the draft runner).
 291:         return self.target_worker.model_runner
 292: 
 293:     @property
      # 注：spec v2 可能同时触碰 target/draft 多个 attention backend，scheduler 用这个集合判断是否需要 CPU seq_lens。
 294:     def spec_v2_attn_backends(self) -> tuple:
 295:         """Attn backends touched by spec_v2 forward; OR-ed by decide_needs_cpu_seq_lens.
 296:         Default returns target only; subclasses extend with draft backends."""
 297:         return (self.target_worker.model_runner.attn_backend,)
 298: 
 299:     @abstractmethod
 300:     def clear_cache_pool(self):
 301:         # TODO: move this abstract method to BaseTpWorker and call through self.model_runner
 302:         pass
 303: 
 304:     def alloc_memory_pool(self, **kwargs):
 305:         pass
 306: 
 307:     def init_attention_backends(self):
 308:         pass
 309: 
 310:     def init_cuda_graphs(self):
 311:         pass
 312: 
 313:     def on_verify_complete_cpu(
      # 注：verify 接受长度回到 CPU 后才触发该 hook，避免 worker 热路径主动制造 GPU->CPU 同步。
 314:         self, num_correct_drafts_per_req: list[int], batch_size: int = 0
 315:     ) -> None:
 316:         """Hook called after verify finishes and accept counts are on CPU.
 317: 
 318:         Default no-op. Adaptive-aware workers override this to feed the
 319:         controller without forcing a GPU→CPU sync in the worker hot path.
 320:         """
 321:         pass
 322: 
 323:     def activate_step_by_batch(self, batch_size: int) -> None:
 324:         """Activate the optimal adaptive step for the current batch size.
 325: 
 326:         Default no-op. Adaptive-aware workers override this to switch
 327:         the runtime state before each draft round.
 328:         """
 329:         pass
```

**读这段时抓住：**

- `target_worker` 和 `draft_worker` 是抽象属性；算法可以共享 target runner，也可以有独立 draft runner。
- `resolve_model_worker_batch` 默认按 target batch 解析，复杂算法会覆盖以适配 draft/verify 的 metadata。
- `finalize_after_verify` 是 accept counts 回到 CPU 后的 hook，用于更新 KV 或算法运行时状态。


### `EAGLEWorkerV2.forward_batch_generation`：EAGLE 的主调度闭环

```python
# python/sglang/srt/speculative/eagle_worker_v2.py:1070-1159
1070:     def forward_batch_generation(self, batch: ScheduleBatch, on_publish=None):
1071:         if batch.forward_mode.is_extend() or batch.is_extend_in_batch:
      # 注：extend/prefill 不是先 draft；target 先处理新 prompt token，并捕获 hidden states 给 draft prefill 用。
1072:             # Target prefill
1073:             target_capture_mode = (
1074:                 CaptureHiddenMode.NULL
1075:                 if self.speculative_algorithm.is_standalone()
1076:                 else CaptureHiddenMode.FULL
1077:             )
1078:             batch.capture_hidden_mode = target_capture_mode
1079:             batch_output = self.target_worker.forward_batch_generation(batch)
      # 注：`batch_output` 接收 `self.target_worker.forward_batch_generation` 的结果：用 target model 执行 verify 或普通生成 forward。
1080: 
1081:             # Spec_v2 convention: batch.seq_lens = length BEFORE this iter's tokens.
1082:             # Extend processed L prompt tokens; next verify iter expects same L.
1083:             batch_output.new_seq_lens = batch.seq_lens
1084:             # Publish before draft_extend so the fence is at target-end.
1085:             if on_publish is not None:
      # 注：overlap 的 publish 点放在 target prefill 之后、draft prefill 之前，让 scheduler 先看到本轮 seq_lens。
1086:                 on_publish(batch_output.new_seq_lens)
1087: 
1088:             # Draft prefill
1089:             with (
1090:                 self.draft_worker.draft_tp_context(
1091:                     self.draft_worker.draft_runner.tp_group
1092:                 ),
1093:                 speculative_moe_backend_context(),
1094:                 speculative_moe_a2a_backend_context(),
1095:                 spec_stage_span("draft_extend"),
1096:             ):
1097:                 batch_output.next_draft_input = (
      # 注：draft prefill 消费 target hidden states 与 next token，填好 draft KV，并返回下一轮 decode 的 `next_draft_input`。
1098:                     self.draft_worker._draft_extend_for_prefill(
1099:                         batch,
1100:                         batch_output.logits_output.hidden_states,
1101:                         batch_output.next_token_ids,
1102:                         batch_output.logits_output.mm_input_embeds,
1103:                     )
1104:                 )
1105:                 return batch_output
1106:         else:
1107:             self.activate_step_by_batch(batch.seq_lens.shape[0])
1108: 
1109:             if batch.spec_info is None:
1110:                 capture_mode = (
1111:                     CaptureHiddenMode.NULL
1112:                     if self.speculative_algorithm.is_standalone()
1113:                     else CaptureHiddenMode.LAST
1114:                 )
1115:                 batch.spec_info = EagleDraftInput.create_idle_input(
1116:                     device=self.device,
1117:                     hidden_size=EagleDraftInput.hidden_size_for(self.draft_worker),
1118:                     dtype=EagleDraftInput.dtype_for(self.draft_worker),
1119:                     topk=self.topk,
1120:                     capture_hidden_mode=capture_mode,
1121:                     vocab_size=self.target_worker.model_config.vocab_size,
1122:                 )
1123:             if self.speculative_num_steps == 0:
      # 注：adaptive spec 可把 draft 步数降到 0；此时仍走 verify 框架，但只采一个 target bonus token。
1124:                 # Drafting disabled (high batch size). _draft_extend below still
1125:                 # runs, keeping draft KV warm for when the batch shrinks.
1126:                 verify_input = self._build_trivial_verify_input(batch)
1127:             else:
1128:                 with (
1129:                     self.draft_worker.draft_tp_context(
1130:                         self.draft_worker.draft_runner.tp_group
1131:                     ),
1132:                     speculative_moe_backend_context(),
1133:                     speculative_moe_a2a_backend_context(),
1134:                     spec_stage_span("draft"),
1135:                 ):
1136:                     verify_input: EagleVerifyInput = self.draft_worker.draft(batch)
      # 注：decode 主路径在这里真正生成 draft 树；输出会立刻变成 batch.spec_info 供 target verify 使用。
1137:             assert verify_input.is_verify_input()
1138:             batch.spec_info = verify_input
1139:             batch_output = self.verify(batch)
      # 注：`verify` 是 EAGLE decode 的 target 检查阶段，返回 accept_lens、new_seq_lens 和下一轮 draft 输入。
1140:             # Publish before draft_extend so the fence is at verify-end.
1141:             if on_publish is not None:
      # 注：verify 完成后立即发布 new_seq_lens；后面的 draft_extend 可以和 scheduler 的下一轮准备重叠。
1142:                 on_publish(batch_output.new_seq_lens)
1143:             if (
1144:                 self.speculative_num_steps == 0
1145:                 and envs.SGLANG_SPEC_SKIP_ZERO_STEP_DRAFT_EXTEND.get()
1146:             ):
1147:                 self._stub_skipped_draft_extend(batch, batch_output)
1148:             else:
1149:                 with (
1150:                     self.draft_worker.draft_tp_context(
1151:                         self.draft_worker.draft_runner.tp_group
1152:                     ),
1153:                     speculative_moe_backend_context(),
1154:                     speculative_moe_a2a_backend_context(),
1155:                     spec_stage_span("draft_extend"),
1156:                 ):
1157:                     self.draft_worker._draft_extend_for_decode(batch, batch_output)
      # 注：draft_extend 用 verify 结果更新 draft 侧运行状态，使下一轮 draft 从最新 accepted token 继续。
1158: 
1159:             return batch_output
```

**读这段时抓住：**

- extend/prefill 分支先跑 target，再用 target hidden + next token 做 draft prefill，给下一轮 decode 准备 draft 状态。
- decode 分支先 draft，再把 `EagleVerifyInput` 放回 `batch.spec_info` 交给 target verify。
- `on_publish` 在 target/verify 结束后、draft_extend 前发布 new_seq_lens，是 overlap scheduler 能提前工作的同步点。
- adaptive spec 可以把 `speculative_num_steps` 降到 0，此时构造 trivial verify input 退化成普通 decode。


### 机制小结：EAGLE draft 如何一次预测多个 token

EAGLE 的 draft 不是让小模型从 prompt 重新自回归一遍，而是复用 target 已经算出的 hidden states 作为起点。prefill/extend 阶段 target 先正常前向，得到 `target_hidden_states` 和 target 采样出的 `next_token_ids`；随后 `draft_extend_for_prefill` 把这些信息喂给 draft model，让 draft KV cache 与当前请求状态对齐。decode 阶段 `EagleDraftWorker.draft_forward` 才能直接从上一轮保存的 `topk_p/topk_index/hidden_states` 继续展开候选。

`draft_forward` 的核心循环是 `for i in range(self.speculative_num_steps)`。第 0 步的输入来自上一轮 target/draft 准备好的 top-k token 与 hidden states；后续每一步都会把上一步 draft model 输出的 logits 归一化，再取 `fast_topk` 或 `fast_sample` 作为下一层候选。也就是说，draft model 仍然按自回归方式一层层预测未来 token，只是这些未来 token 先停在 speculative buffer 里，暂时还没有提交给请求输出。

当 `topk == 1` 时，每个请求每一步只有一个候选，拓扑就是一条链：第 1 个 draft token 的父节点是 bonus/root token，第 2 个 draft token 的父节点是第 1 个 draft token，依此类推。因此代码可以直接拼接 `token_list`，并复用预分配的 `parent_list/top_scores_index`。

当 `topk > 1` 时，候选会形成树。直观地说，第 1 层有 top-k 个 token；第 2 层会从第 1 层的每个候选继续扩展 top-k，于是产生 top-k × top-k 个“父 token -> 子 token”组合；更深层同理。`select_top_k_tokens` 在每一层不仅返回要送入 draft model 的 `input_ids`，还返回 `tree_info`：候选累计分数、候选 token id、以及每个候选对应的父节点位置。循环结束后，`organize_draft_results` 会把多层候选拉平成一批节点，按累计分数选出 `speculative_num_draft_tokens - 1` 个最有希望的节点，再保留父子关系。最后 `build_tree_kernel_efficient` 把这些节点转换成 target verify 所需的 `draft_token`、tree attention mask、`retrieve_index`、`retrieve_next_token` 和 `retrieve_next_sibling`。

因此，EAGLE tree draft 的本质是：draft model 负责提出一棵高概率候选树；树上的每个节点都是“某条候选路径上的下一个 token”；父子边记录这个 token 应该接在哪个历史候选后面。target model 随后会一次性验证这棵树，而不是逐条路径重复前向。


### `EAGLEWorkerV2.verify`：target 检查 draft 树并产出统一 batch result

```python
# python/sglang/srt/speculative/eagle_worker_v2.py:1431-1584
1431:     def verify(self, batch: ScheduleBatch):
1432:         fwd_stream = torch.get_device_module(self.device).current_stream()
1433:         verify_input: EagleVerifyInput = batch.spec_info
1434:         record_stream_for_v2_verify(batch, verify_input, fwd_stream)
1435: 
1436:         verify_input.num_tokens_per_req = self.speculative_num_steps + 1
1437:         bs = len(batch.seq_lens)
1438: 
1439:         # Batch 1: Target verify
1440:         # Prepare for target verify in a separate stream
1441:         with self.plan_stream_ctx:
      # 注：verify 的 batch 构造放到 plan stream，尽量和主 compute stream 分离准备 tree mask / positions / out_cache_loc。
1442:             verify_forward_batch, can_run_cuda_graph = eagle_prepare_for_verify(
1443:                 verify_input,
1444:                 self.req_to_token_pool,
1445:                 batch,
1446:                 self.target_worker,
1447:             )
1448: 
1449:         # Cover post-prepare rebinds: draft_token, plan_stream-allocated out_cache_loc.
1450:         record_stream_each((batch.input_ids, batch.out_cache_loc), fwd_stream)
1451: 
1452:         # Correct some buffers due to the overlap plan
1453:         if self.plan_stream:
1454:             torch.get_device_module(self.device).current_stream().wait_stream(
1455:                 self.plan_stream
1456:             )
1457:             if (
1458:                 _is_npu
1459:                 and self._target_worker.model_runner.model_is_mrope
1460:                 and batch.spec_info is not None
1461:                 and getattr(batch.spec_info, "positions", None) is not None
1462:                 and not batch.forward_mode.is_idle()
1463:             ):
1464:                 # mrope_position depends on draft output in default stream and is computed in plan stream,
1465:                 # causing errors. Compute it here for correct values.
1466:                 verify_forward_batch.compute_spec_mrope_positions(
1467:                     self._target_worker.model_runner, batch
1468:                 )
1469: 
1470:             # Some values such as custom_mask and position depend on the output of draft,
1471:             # so the previous plan step used the wrong values. Here, we need to run the related
1472:             # computation again to update them to the correct values.
1473:             self.target_worker.model_runner.attn_backend.update_verify_buffers_to_fill_after_draft(
1474:                 verify_input,
1475:                 (
1476:                     self.target_worker.model_runner.decode_cuda_graph_runner.bs
1477:                     if can_run_cuda_graph
1478:                     else None
1479:                 ),
1480:             )
1481: 
1482:         # Prepare grammar data on CPU if needed
1483:         if batch.has_grammar:
1484:             retrieve_next_token_cpu = verify_input.retrieve_next_token.cpu()
1485:             retrieve_next_sibling_cpu = verify_input.retrieve_next_sibling.cpu()
1486:             draft_tokens_cpu = verify_input.draft_token.view(
1487:                 verify_input.retrieve_next_token.shape
1488:             ).cpu()
1489: 
1490:         # Run target verify batch in the main compute stream (GPU compute).
1491:         # Metadata init is skipped iff cuda-graph already ran load_batch —
1492:         # eagle_prepare_for_verify marked the batch in exactly that case; the
1493:         # non-cuda-graph path stays unmarked and gets forward_extend's init
1494:         # (post-pad).
1495:         forward_batch_output = self.target_worker.forward_batch_generation(
      # 注：target verify forward 使用的是刚构造好的 verify_forward_batch，而不是原始 scheduler batch。
1496:             batch=None,
1497:             forward_batch=verify_forward_batch,
1498:             is_verify=True,
1499:         )
1500:         logits_output = forward_batch_output.logits_output
1501: 
1502:         # Generate vocab mask for constrained decoding
1503:         vocab_mask = None
1504:         if batch.has_grammar:
1505:             # Generate the logit mask for structured output.
1506:             vocab_mask = generate_token_bitmask(
1507:                 batch.reqs,
1508:                 verify_input,
1509:                 retrieve_next_token_cpu,
1510:                 retrieve_next_sibling_cpu,
1511:                 draft_tokens_cpu,
1512:                 batch.sampling_info.vocab_size,
1513:             )
1514: 
1515:             if vocab_mask is not None:
1516:                 assert verify_input.grammar is not None
1517:                 vocab_mask = vocab_mask.to(verify_input.retrieve_next_token.device)
1518:                 # NOTE: otherwise, this vocab mask will be the one from the previous extend stage
1519:                 # and will be applied to produce wrong results
1520:                 batch.sampling_info.vocab_mask = None
1521: 
1522:         # Sample
1523:         maybe_detect_nan(logits_output.next_token_logits, "verify: target model logits")
1524:         maybe_detect_inf(logits_output.next_token_logits, "verify: target model logits")
1525:         (
      # 注：采样阶段把 target logits 投影回 draft 树，得到每个请求最终接受的路径和长度。
1526:             predict,
1527:             accept_lens,
1528:             accept_index,
1529:         ) = eagle_sample(verify_input, batch, logits_output, vocab_mask)
1530:         new_seq_lens = batch.seq_lens + accept_lens
1531: 
1532:         # Update mamba state for hybrid GDN models after verification
1533:         commit_mamba_states_after_verify(
1534:             self.target_worker,
1535:             batch,
1536:             accept_lens,
1537:             accept_index,
1538:             self.speculative_num_draft_tokens,
1539:         )
1540: 
1541:         if not batch.forward_mode.is_idle():
1542:             accept_tokens = predict[accept_index]
1543:             bonus_tokens = torch.empty_like(accept_lens, dtype=torch.int32)
1544:             # stride = accept_tokens per-req width = accept_index.shape[1]
1545:             # (spec_steps + 1); NOT num_draft_tokens, wrong for topk > 1 trees.
1546:             fill_bonus_tokens[(bs,)](
1547:                 accept_tokens,
1548:                 accept_lens,
1549:                 bonus_tokens,
1550:                 accept_index.shape[1],
1551:             )
1552:         else:
1553:             bonus_tokens = torch.empty((0,), device=self.device, dtype=torch.int32)
1554: 
1555:         if batch.return_logprob and not batch.forward_mode.is_idle():
1556:             compute_spec_v2_logprobs(
1557:                 batch, logits_output, predict, accept_index, self.speculative_num_steps
1558:             )
1559: 
1560:         if not batch.forward_mode.is_idle() and self.topk > 1:
      # 注：topk>1 的 accepted path 可能在树中间，需要 compact 到后续 decode 假设的连续前缀位置。
1561:             # topk == 1 needs nothing here: the accepted path is already the front
1562:             # chain, so the whole compaction is an identity transform.
1563:             predict = self._finalize_accept_tree_path(
1564:                 batch, accept_index, accept_lens, predict, logits_output, bs
1565:             )
1566: 
1567:         next_draft_input = EagleDraftInput(bonus_tokens=bonus_tokens)
1568: 
1569:         # verify_forward_batch transitively holds verify-time GPU tensors
1570:         # (draft_token / out_cache_loc / ...) that must outlive the imminent
1571:         # batch.input_ids rebind in prepare_for_draft_extend.
1572:         # Scheduler pins it in batch_record_buf for the 2-iter window.
1573:         return GenerationBatchResult(
      # 注：`GenerationBatchResult` 是 scheduler 的统一接口；speculative 结果在这里被整理成普通 decode 可消费的形状。
1574:             logits_output=logits_output,
1575:             next_token_ids=predict,
1576:             can_run_cuda_graph=can_run_cuda_graph,
1577:             speculative_num_draft_tokens=self.speculative_num_draft_tokens,
1578:             next_draft_input=next_draft_input,
1579:             accept_lens=accept_lens,
1580:             new_seq_lens=new_seq_lens,
1581:             routed_experts_output=forward_batch_output.routed_experts_output,
1582:             indexer_topk_output=forward_batch_output.indexer_topk_output,
1583:             extra_keep_alive_refs=[verify_forward_batch],
1584:         )
```

**读这段时抓住：**

- `eagle_prepare_for_verify` 把 draft 树转成 target forward batch，包含 tree mask、positions 和 KV 写入位置。
- `target_worker.forward_batch_generation(..., is_verify=True)` 是真正的 target verify forward。
- `eagle_sample` 根据 target logits 决定接受路径，输出 `predict/accept_lens/accept_index`。
- topk>1 的树状 draft 需要把 accepted path compact 到每请求 block 前部，否则后续 chain-layout 代码会读错 KV/hidden。
- 返回的 `GenerationBatchResult` 把 speculative 结果伪装成普通 decode 结果，scheduler 后处理可以复用同一套路径。


### 机制小结：target 如何 verify draft，以及拒绝后如何纠错

verify 的第一步是 `eagle_prepare_for_verify`。它把 draft 阶段产生的树状候选改写成 target model 能一次前向的 batch：`input_ids` 是树中所有候选节点的 token id，`positions` 是这些候选在各自请求中的逻辑位置，`custom_mask` 限制每个节点只能看到原始上下文和自己的祖先节点，`out_cache_loc` 则为 target verify 的临时 KV 写入位置。这样 target model 对整棵 draft 树做一次 forward，就能得到每个候选节点后的 next-token logits。

如果采样配置是全 greedy，`eagle_sample` 会把 target logits 取 `argmax`，再调用 `verify_tree_greedy_func` 沿树检查：当前 draft token 是否等于 target 在父节点处预测的 top-1 token。相等就接受该节点，并沿这个节点继续向下；不相等就停止，当前位置改用 target 的预测 token。树状 draft 的关键在 `retrieve_index/retrieve_next_token/retrieve_next_sibling`：它们告诉 kernel 当前接受节点在扁平 buffer 中的位置、它的第一个孩子在哪里、下一个兄弟在哪里。kernel 不是线性扫完整棵树，而是在树结构中选择一条被 target 认可的路径。

非 greedy 采样下，target logits 会先应用 temperature、top-k、top-p，并在有 grammar 时通过 `vocab_mask` 屏蔽非法 token，然后得到 `target_probs`。如果启用 classic rejection sampling，draft 还必须提供同 vocab 的 `draft_probs`。每个 draft token 的接受概率由 target 概率 `p` 和 draft proposal 概率 `q` 共同决定：代码中的条件是 `coin * q < p`，等价于以 `min(1, p/q)` 的概率接受。target 越相信这个 draft token，越容易接受；draft 高估而 target 低估的 token 会更容易被拒绝。

拒绝并不是简单地“重新跑一遍 decode”。一旦某个 draft token 被拒绝，kernel 会从修正分布 `max(target_probs - draft_probs, 0)` 中采样一个替代 token，写到最后一个已接受节点之后。这就是 speculative sampling 的纠错机制：接受的前缀来自 draft，但第一个不被 target 认可的位置由 target 的残差分布补上，从而保持整体输出分布等价于 target model。若所有 draft token 都被接受，kernel 还会从 target 当前分布中额外采一个 bonus token，因此返回给 scheduler 的 `accept_lens` 是“接受 draft 数 + 1 个 target bonus”。

grammar/structured output 的约束发生在 verify 采样前：`generate_token_bitmask` 生成词表 mask，`apply_vocab_mask` 把非法 token 的 target logits 屏蔽掉。这样非法 draft token 即使出现在候选树里，也不会被 target 的合法分布支持；greedy 路径会因为 target top-1 不匹配而停下，采样路径会因为非法 token 的 target 概率接近 0 而被拒绝，然后由 masked target 分布采出合法替代 token。


### 机制深挖：`topk > 1` 时 target 不是验证整棵树是否全对，而是在树里找一条可接受路径

先把关键点说清楚：target model 并不会要求 draft 树上的每个分支都正确。`topk > 1` 的意义是让 draft 在同一层给出多个备选分支；target verify 的任务是在这些兄弟分支中找一个能接上 target 预测/采样结果的孩子，然后沿这个孩子继续向下。如果某一层所有兄弟都接不上，就停止接受，并由 target 自己补出当前位置的 token。

树在 verify 前会被摊平成每个请求一行的 `candidates`。以 `sgl-kernel/tests/speculative/test_eagle_utils.py` 的第一个请求为例：

```text
candidates[0]             = [0, 1, 2, 3, 4, 5]
retrieve_next_token[0]    = [1, 2, -1, 4, 5, -1]
retrieve_next_sibling[0]  = [-1, 3, -1, -1, -1, -1]
retrieve_index[0]         = [0, 1, 2, 3, 4, 5]
```

这四个数组共同表达一棵树：

```text
local node 0: token 0, root/bonus 节点
  child -> local node 1: token 1
    child -> local node 2: token 2
  sibling of local node 1 -> local node 3: token 3
    child -> local node 4: token 4
      child -> local node 5: token 5
```

`retrieve_next_token[i]` 是“第一个孩子”的 local node id；`retrieve_next_sibling[i]` 是“下一个兄弟”的 local node id；`retrieve_index[i]` 是这个 local node 在 target verify 扁平 logits/predict buffer 里的全局行号。注意 local node id 用来走树，global/retrieve index 用来读写 `target_predict`、`predicts`、`accept_index`。

target 一次 forward 这 6 个节点后，会得到 `target_predict` 或 `target_probs`。测试里人为设置了：

```text
target_predict[0, 0] = 3  # target 认为 root 后应该接 token 3
target_predict[0, 3] = 4  # target 认为 node 3 后应该接 token 4
target_predict[0, 4] = 5  # target 认为 node 4 后应该接 token 5
其他位置默认预测 token 18
```

于是 greedy verify 的路径是：

1. 从 root local node 0 开始，先看第一个孩子 local node 1，draft token 是 1。
2. target 在 root 的 logits 行预测 token 3，所以 node 1 不匹配。
3. 沿 `retrieve_next_sibling[1]` 跳到 local node 3，draft token 是 3，匹配 target 的 root 预测，于是接受 node 3。
4. 当前已接受末端变成 node 3；下一层看 node 3 的第一个孩子 local node 4。target 在 node 3 的 logits 行预测 token 4，所以接受 node 4。
5. 再看 node 4 的孩子 local node 5。target 在 node 4 的 logits 行预测 token 5，所以接受 node 5。
6. 最后 `accept_index[0] = [0, 3, 4, 5]`，`accept_token_num[0] = 3`。这里 0 是 root 的全局 index，3/4/5 是被接受 draft 节点的全局 index。

这正是 `VerifyTreeGreedy` C++ kernel 的循环：每一层先 `cur_index = retrieve_next_token[cur_index]` 走到第一个孩子；如果这个孩子 token 不等于 target 当前行的预测，就不断 `cur_index = retrieve_next_sibling[cur_index]` 横向找兄弟；找到匹配的兄弟才写 `predicts[last_accepted_retrieve_idx]`、推进 `accept_index`，并把 `last_accepted_retrieve_idx` 改成这个被接受节点的全局 index。

非 greedy 的 `tree_speculative_sampling_target_only` 也是同一个树遍历骨架，只是“是否接受兄弟节点”的条件从 token id 相等变成概率接受。它会累加同一层已看过兄弟的 target 概率 `prob_acc`，当 `coin <= prob_acc / threshold_acc` 或某个兄弟自己的 `target_prob_single >= threshold_single` 时接受该兄弟；被跳过的兄弟会把 target 概率写进 `draft_probs`，最后从 `relu(target_probs - draft_probs)` 中采样补偿 token。也就是说，tree sampling 仍然是在每一层兄弟集合里挑一条路径，而不是把所有分支都提交。

最后，`EAGLEWorkerV2.verify` 里 `topk > 1` 后还要调用 `_finalize_accept_tree_path`。原因是被接受的路径可能是 `[root, node 3, node 4, node 5]`，并不在每请求 verify block 的前几个连续位置；而后续普通 decode/draft_extend 默认读取“连续链式布局”。所以 `_finalize_accept_tree_path` 会根据 `accept_index` 把接受路径对应的 KV、`predict`、hidden states gather 到每个请求 block 的前部，未接受分支则变成可释放/覆盖的临时 verify KV。


In [ ]:
show_lines("python/sglang/srt/speculative/eagle_utils.py", 407, 529)
show_lines("sgl-kernel/csrc/speculative/eagle_utils.cu", 275, 312)
show_lines("sgl-kernel/csrc/speculative/speculative_sampling.cuh", 64, 104)
show_lines("sgl-kernel/tests/speculative/test_eagle_utils.py", 8, 84)


### `NGRAMWorker.forward_batch_generation`：没有 draft model 的 speculative 路径
 
```python
# python/sglang/srt/speculative/ngram_worker.py:372-521
 372:     def forward_batch_generation(
 373:         self, batch: ScheduleBatch, on_publish=None
 374:     ) -> GenerationBatchResult:
 375:         fwd_stream = torch.get_device_module(self.device).current_stream()
 376:         record_stream_for_v2_verify(batch, None, fwd_stream)
 377:         bs = len(batch.reqs)
 378: 
 379:         set_time_batch(batch.reqs, "set_spec_draft_start_time", trace_only=True)
      # 注：NGRAM 没有 draft model；这里的 draft 时间统计覆盖 CPU corpus 查询和 batch 改写。
 380:         self._prepare_for_speculative_decoding(batch)
 381:         set_time_batch(batch.reqs, "set_spec_draft_end_time", trace_only=True)
 382: 
 383:         verify_input: NgramVerifyInput = batch.spec_info
 384:         accept_lens = torch.ones(bs, dtype=torch.int32, device=self.device)
 385: 
 386:         if batch.forward_mode.is_target_verify():
      # 注：只有 decode 被改写成 TARGET_VERIFY；extend/prefill 仍然走普通 target forward。
 387:             # Prepare grammar data on CPU if needed
 388:             if batch.has_grammar:
 389:                 retrieve_next_token_cpu = verify_input.retrieve_next_token.cpu()
 390:                 retrieve_next_sibling_cpu = verify_input.retrieve_next_sibling.cpu()
 391:                 draft_tokens_cpu = verify_input.draft_token.view(
 392:                     verify_input.retrieve_next_token.shape
 393:                 ).cpu()
 394: 
 395:             batch_result = self.target_worker.forward_batch_generation(
      # 注：NGRAM verify 复用 target worker，候选 token 来自 corpus，但正确性仍由 target logits 决定。
 396:                 batch, is_verify=True
 397:             )
 398: 
 399:             logits_output, can_run_cuda_graph = (
 400:                 batch_result.logits_output,
 401:                 batch_result.can_run_cuda_graph,
 402:             )
 403: 
 404:             verify_input: NgramVerifyInput = batch.spec_info
 405:             vocab_mask = None
 406:             if batch.has_grammar:
 407:                 # Generate the logit mask for structured output.
 408:                 # Overlap the CPU operations for bitmask generation with the forward pass.
 409:                 vocab_mask = generate_token_bitmask(
 410:                     batch.reqs,
 411:                     verify_input,
 412:                     retrieve_next_token_cpu,
 413:                     retrieve_next_sibling_cpu,
 414:                     draft_tokens_cpu,
 415:                     batch.sampling_info.vocab_size,
 416:                 )
 417: 
 418:                 if vocab_mask is not None:
 419:                     assert verify_input.grammar is not None
 420:                     vocab_mask = vocab_mask.to(verify_input.retrieve_next_token.device)
 421:                     # NOTE (sk): otherwise, this vocab mask will be the one from the previous extend stage
 422:                     # and will be applied to produce wrong results
 423:                     batch.sampling_info.vocab_mask = None
 424: 
 425:             # Sample
 426:             maybe_detect_nan(
 427:                 logits_output.next_token_logits, "verify: target model logits"
 428:             )
 429:             maybe_detect_inf(
 430:                 logits_output.next_token_logits, "verify: target model logits"
 431:             )
 432:             (
      # 注：这里复用 EAGLE 的采样工具：输入同样是 verify_input + target logits + 可选 grammar mask。
 433:                 predict,
 434:                 accept_lens,
 435:                 accept_index,
 436:             ) = eagle_sample(verify_input, batch, logits_output, vocab_mask)
 437:             new_seq_lens = batch.seq_lens + accept_lens
 438:             commit_mamba_states_after_verify(
 439:                 self.target_worker,
 440:                 batch,
 441:                 accept_lens,
 442:                 accept_index,
 443:                 self.draft_token_num,
 444:             )
 445:             accept_tokens = predict[accept_index].flatten()
 446:             next_token_ids = accept_tokens
 447: 
 448:             # The KV mover expects drafts-only counts. NGRAM's
 449:             # accept_lens includes the bonus token, matching scheduler output.
 450:             num_correct_drafts_per_req = accept_lens - 1
 451:             move_accept_tokens_to_target_kvcache(
      # 注：接受路径对应的 KV 需要搬到 target cache 的连续位置，后续普通 decode 才能直接接上。
 452:                 batch,
 453:                 accept_index,
 454:                 num_correct_drafts_per_req,
 455:                 self.token_to_kv_pool_allocator,
 456:             )
 457:             if batch.return_logprob:
 458:                 # The last arg is the accept_index row width minus 1. NGRAM's
 459:                 # accept_index is (bs, draft_token_num) -- the tree depth is not
 460:                 # bounded by spec_steps like EAGLE's (bs, spec_steps + 1).
 461:                 compute_spec_v2_logprobs(
 462:                     batch,
 463:                     logits_output,
 464:                     predict,
 465:                     accept_index,
 466:                     self.draft_token_num - 1,
 467:                 )
 468: 
 469:             if on_publish is not None:
 470:                 on_publish(new_seq_lens)
 471: 
 472:             self._update_ngram_corpus(batch)
      # 注：NGRAM corpus 在 verify 后更新，下一轮检索才能利用刚接受的新 token。
 473:             # Erase match state of requests that left the decode batch.
 474:             # req.finished() is unusable here: under overlap it flips at result
 475:             # processing, one iteration after the request left the batch.
 476:             # The last batch's entries persist while idle (bounded, small).
 477:             cur_rids = {req.rid for req in batch.reqs}
 478:             departed_rids = self._prev_decode_rids - cur_rids
 479:             if departed_rids:
 480:                 self.ngram_corpus.erase_match_state(list(departed_rids))
 481:             self._prev_decode_rids = cur_rids
 482:             batch.forward_mode = ForwardMode.DECODE
 483: 
 484:         else:
 485:             batch_result = self.target_worker.forward_batch_generation(batch)
      # 注：model worker 返回的生成结果，后续 scheduler 会据此更新 token、finish reason、KV cache 和回包。
 486:             logits_output, predict, can_run_cuda_graph = (
 487:                 batch_result.logits_output,
 488:                 batch_result.next_token_ids,
 489:                 batch_result.can_run_cuda_graph,
 490:             )
 491:             new_seq_lens = batch.seq_lens.clone()
 492: 
 493:             accept_tokens = torch.zeros(
 494:                 bs, self.draft_token_num, dtype=torch.int32, device=self.device
 495:             )
 496:             accept_tokens[:, 0] = predict
 497:             accept_tokens = accept_tokens.flatten()
 498:             next_token_ids = predict
 499: 
 500:             if on_publish is not None:
 501:                 on_publish(new_seq_lens)
 502: 
 503:         # Construct the next draft input
 504:         next_draft_input = NgramVerifyInput(
 505:             draft_token_num=self.draft_token_num,
 506:             new_seq_lens=new_seq_lens,
 507:             accept_tokens=accept_tokens,
 508:             accept_lens=accept_lens,
 509:         )
 510:         return GenerationBatchResult(
 511:             logits_output=logits_output,
 512:             next_token_ids=next_token_ids,
 513:             can_run_cuda_graph=can_run_cuda_graph,
 514:             accept_lens=accept_lens,
 515:             # Consumed by the non-overlap V2 scheduler branch to advance
 516:             # batch.seq_lens after the isolation restore; overlap mode relays
 517:             # it via on_publish instead.
 518:             new_seq_lens=new_seq_lens,
 519:             next_draft_input=next_draft_input,
 520:             speculative_num_draft_tokens=self.speculative_num_draft_tokens,
 521:         )
```

**读这段时抓住：**

- `_prepare_for_speculative_decoding` 从 ngram corpus 找候选，并把 batch 改写成 `TARGET_VERIFY`。
- target verify、`eagle_sample`、KV 搬移和 logprob 对齐仍复用 EAGLE 的通用 verify 工具。
- `_update_ngram_corpus` 在 verify 后写入新接受 token，让下一轮 ngram 检索看到最新上下文。
- NGRAM 的 `next_draft_input` 只需要保存 accept_tokens/accept_lens/new_seq_lens，不需要 hidden states 或 draft KV。


### 机制小结：NGRAM 没有 draft model，draft 输入从哪里来

NGRAM speculative decoding 把“提出候选”这件事从神经网络换成了检索。`NGRAMWorker` 维护一个 `ngram_corpus`，里面记录请求历史中出现过的 token 片段。每轮 decode 前，`_prepare_draft_tokens` 会取当前请求的最近一段上下文作为 query：它把 `origin_input_ids`、已输出的 `output_ids`，以及 overlap 模式下上一轮刚接受但还没写回 `req.output_ids` 的 token 拼起来，再截取最长 `max_trie_depth` 的后缀。

随后 `ngram_corpus.batch_get(req_ids, batch_tokens, total_lens)` 查找这个后缀是否在历史语料中出现过。如果出现过，就把历史中紧随其后的 token 作为 draft token；如果能找到多个延续，返回的 `mask` 会描述这些候选之间的树状关系。这里没有 draft logits、没有 draft hidden states，也没有 draft KV cache；NGRAM 只提供 `draft_token` 和 tree mask，后续仍交给 target verify。

`_prepare_for_speculative_decoding` 会把检索结果拷到 GPU buffer，调用 `reconstruct_indices_from_tree_mask` 生成 `retrieve_index/retrieve_next_token/retrieve_next_sibling/positions`，再把 batch 改成 `ForwardMode.TARGET_VERIFY`。从这一刻开始，NGRAM 和 EAGLE 共用同一套 verify 思路：target 对候选树做前向，`eagle_sample` 决定接受路径，接受后的 KV 被搬到 target cache 的提交位置。

NGRAM 的正确性完全依赖 target verify，而不是依赖 ngram 检索一定准确。检索命中只是“猜测未来 token 可能重复历史片段”；猜错时 target 会拒绝并补出 target 分布下的 token。verify 结束后 `_update_ngram_corpus` 会把本轮接受的 token 写回 corpus，所以 NGRAM 的候选来源会随着请求继续生成而增长。


## Accept/reject 的正确性

Greedy verify 可以比较 draft token 与 target top-1；采样场景更复杂，需要 rejection sampling，确保输出分布仍等价于 target model。`reject_sampling.py` 里实现的是接受路径与最终 token 的概率修正。


In [ ]:
show_lines("python/sglang/srt/speculative/reject_sampling.py", 1, 156)


### `reject_sampling.py`：拒绝采样的核心分支

```python
# python/sglang/srt/speculative/reject_sampling.py:6-156
   6: def speculative_sampling_classic_kernel(
   7:     # Pointers
   8:     Predicts,
   9:     AcceptIndex,
  10:     AcceptTokenNum,
  11:     Candidates,
  12:     RetriveIndex,
  13:     UniformSamples,
  14:     UniformSamplesFinal,
  15:     TargetProbs,
  16:     DraftProbs,
  17:     # Strides
  18:     stride_cand_b,
  19:     stride_cand_s,
  20:     stride_idx_b,
  21:     stride_idx_s,
  22:     stride_uni_b,
  23:     stride_uni_s,
  24:     stride_tp_b,
  25:     stride_tp_s,
  26:     stride_tp_v,
  27:     stride_dp_b,
  28:     stride_dp_s,
  29:     stride_dp_v,
  30:     # Constants
  31:     NUM_SLOTS: tl.constexpr,
  32:     VOCAB_SIZE: tl.constexpr,
  33:     BLOCK_V: tl.constexpr,
  34: ):
  35:     pid = tl.program_id(0)
  36:     cur_prob_row = 0
  37: 
  38:     cand_ptr_base = Candidates + pid * stride_cand_b
  39:     idx_ptr_base = RetriveIndex + pid * stride_idx_b
  40:     uni_ptr_base = UniformSamples + pid * stride_uni_b
  41: 
  42:     root_global_idx = tl.load(idx_ptr_base + 0 * stride_idx_s)
  43:     tl.store(AcceptIndex + pid * stride_idx_b + 0 * stride_idx_s, root_global_idx)
  44:     last_accepted_global_idx = root_global_idx
  45: 
  46:     num_accept = 0
  47: 
  48:     # Verification Loop
  49:     step = 1
  50:     continue_verifying = 1
  51: 
  52:     while (step < NUM_SLOTS) and (continue_verifying == 1):
  53:         draft_token = tl.load(cand_ptr_base + step * stride_cand_s)
  54: 
  55:         offset_prob = (
  56:             (pid * stride_tp_b)
  57:             + (cur_prob_row * stride_tp_s)
  58:             + (draft_token * stride_tp_v)
  59:         )
  60:         offset_draft = (
  61:             (pid * stride_dp_b)
  62:             + (cur_prob_row * stride_dp_s)
  63:             + (draft_token * stride_dp_v)
  64:         )
  65: 
  66:         p = tl.load(TargetProbs + offset_prob)
  67:         q = tl.load(DraftProbs + offset_draft)
  68: 
  69:         coin = tl.load(uni_ptr_base + (step - 1) * stride_uni_s)
  70: 
  71:         if coin * q < p:
  72:             num_accept += 1
  73:             cur_prob_row = step
  74:             tl.store(Predicts + last_accepted_global_idx, draft_token)
  75: 
  76:             curr_global_idx = tl.load(idx_ptr_base + step * stride_idx_s)
  77:             tl.store(
  78:                 AcceptIndex + pid * stride_idx_b + num_accept * stride_idx_s,
  79:                 curr_global_idx,
  80:             )
  81:             last_accepted_global_idx = curr_global_idx
  82: 
  83:             step += 1
  84:         else:
  85:             continue_verifying = 0
  86: 
  87:     tl.store(AcceptTokenNum + pid, num_accept)
      # 注：`AcceptTokenNum` 写入本请求接受的 draft 数，scheduler 后续用它更新 accept_lens 和请求输出。
  88: 
  89:     # Final Sampling
  90:     all_drafts_accepted = continue_verifying
  91:     coin_final = tl.load(UniformSamplesFinal + pid)
  92:     norm_sum = 0.0
  93: 
  94:     tp_base_ptr = TargetProbs + (pid * stride_tp_b) + (cur_prob_row * stride_tp_s)
  95:     # DraftProbs has only num_steps rows (TargetProbs has num_steps + 1). When
  96:     # all drafts are accepted cur_prob_row == num_steps is out of bounds for
  97:     # DraftProbs, but the all-accepted branch samples pure target p and never
  98:     # dereferences this pointer; on rejection cur_prob_row <= num_steps - 1.
  99:     dp_base_ptr_safe = DraftProbs + (pid * stride_dp_b) + (cur_prob_row * stride_dp_s)
 100: 
 101:     # Pass 1: Sum
 102:     for v_start in range(0, VOCAB_SIZE, BLOCK_V):
 103:         v_offsets = v_start + tl.arange(0, BLOCK_V)
 104:         mask = v_offsets < VOCAB_SIZE
 105: 
 106:         p_ptr = tp_base_ptr + v_offsets * stride_tp_v
 107:         p_val = tl.load(p_ptr, mask=mask, other=0.0)
 108: 
 109:         if all_drafts_accepted:
 110:             val = p_val
 111:         else:
 112:             q_ptr = dp_base_ptr_safe + v_offsets * stride_dp_v
 113:             q_val = tl.load(q_ptr, mask=mask, other=0.0)
 114:             diff = p_val - q_val
 115:             val = tl.where(diff > 0.0, diff, 0.0)
 116: 
 117:         norm_sum += tl.sum(val)
 118: 
 119:     # Pass 2: CDF. Degenerate residual (norm_sum == 0, i.e. p == q everywhere on
 120:     # rejection) leaves the cumsum at 0 <= target_u, so final_token falls back to
 121:     # VOCAB_SIZE - 1; acceptable since this case is numerically near-impossible.
 122:     target_u = coin_final * norm_sum
 123:     cum_sum = 0.0
 124:     final_token = VOCAB_SIZE - 1
 125:     found = 0
 126: 
 127:     for v_start in range(0, VOCAB_SIZE, BLOCK_V):
 128:         if found == 0:
 129:             v_offsets = v_start + tl.arange(0, BLOCK_V)
 130:             mask = v_offsets < VOCAB_SIZE
 131: 
 132:             p_ptr = tp_base_ptr + v_offsets * stride_tp_v
 133:             p_val = tl.load(p_ptr, mask=mask, other=0.0)
 134: 
 135:             if all_drafts_accepted:
 136:                 val = p_val
 137:             else:
 138:                 q_ptr = dp_base_ptr_safe + v_offsets * stride_dp_v
 139:                 q_val = tl.load(q_ptr, mask=mask, other=0.0)
 140:                 diff = p_val - q_val
 141:                 val = tl.where(diff > 0.0, diff, 0.0)
 142: 
 143:             block_cumsum = tl.cumsum(val, axis=0)
 144:             total_cumsum = cum_sum + block_cumsum
 145: 
 146:             candidates_mask = total_cumsum > target_u
 147:             has_match = tl.max(candidates_mask, axis=0)
 148: 
 149:             if has_match:
 150:                 match_idx = tl.argmax(candidates_mask.to(tl.int32), axis=0)
 151:                 final_token = v_start + match_idx
 152:                 found = 1
 153: 
 154:             cum_sum += tl.sum(val)
 155: 
 156:     tl.store(Predicts + last_accepted_global_idx, final_token)
      # 注：最终 token 写回 `Predicts`，位置是最后一个已接受 draft 的全局位置；scheduler 之后像普通 decode 一样消费它。
```

**读这段时抓住：**

- kernel 逐步比较 draft token 在 target/draft 概率下是否可接受，接受则推进 `last_accepted_global_idx`。
- 一旦拒绝，最终 token 不再直接用 draft，而从修正后的 target-minus-draft 分布采样。
- 全部接受时还要采一个 bonus token，这就是 accept_lens 往往包含 bonus 的原因。


### 机制小结：`speculative_sampling_classic_kernel` 的逐步过程

这个 Triton kernel 处理的是“链式候选”的 classic rejection sampling。grid 的每个 program 对应一个请求；`Candidates[pid, :]` 是该请求的候选 token 序列，`RetriveIndex[pid, :]` 是这些候选在 verify buffer 中对应的全局位置，`TargetProbs[pid, step, vocab]` 是 target verify 后的概率分布，`DraftProbs[pid, step, vocab]` 是 draft proposal 分布。

kernel 先把 root 的全局位置写入 `AcceptIndex`。这里 root 不是一个新生成 token，而是 verify 路径的起点，用来记录后续 token 应该接在哪个全局节点后面。`last_accepted_global_idx` 始终指向当前已接受路径的末端，后面写 `Predicts` 和最终 token 都依赖它定位。

接着进入 verification loop，从 `step = 1` 开始逐个检查 draft token。对当前 draft token，kernel 在 target 分布中取 `p = TargetProbs[..., draft_token]`，在 draft 分布中取 `q = DraftProbs[..., draft_token]`，再读一个均匀随机数 `coin`。如果 `coin * q < p`，这个 token 被接受：`Predicts[last_accepted_global_idx]` 写入该 token，`AcceptIndex` 追加当前节点的全局位置，`num_accept` 加一，并把 `cur_prob_row` 推进到当前 step。否则 verification loop 停止，后续 draft token 全部丢弃。

loop 结束后，`AcceptTokenNum[pid]` 保存接受了多少个 draft token。然后 kernel 必须补一个最终 token。如果所有 draft 都被接受，补 token 直接从 target 当前分布 `p` 中采样，也就是 speculative decoding 常说的 bonus token。如果中途拒绝，补 token 从残差分布 `max(p - q, 0)` 中采样。这个残差分布正是 classic speculative sampling 保持 target 分布不变的关键：draft 已经“占用”了一部分 proposal 概率，拒绝后只能从 target 相对 draft 还缺的那部分概率质量里补。

最后的采样分两遍扫 vocab。第一遍按 block 累加 `norm_sum`，得到目标分布或残差分布的总概率质量；第二遍用 `coin_final * norm_sum` 做 CDF 查找，找到最终 token id，并写入 `Predicts[last_accepted_global_idx]`。因此 kernel 的输出包括三类信息：接受了几个 draft token、接受路径在 verify buffer 中的节点位置、以及每个已接受位置后应该提交给请求的 token。


## KV cache 为什么是难点

Speculative decoding 不只是多生成几个 token。draft token 可能被拒绝，因此 KV 写入需要区分：

- draft KV 是否真实写入。
- target verify 使用哪段 cache loc。
- 接受后哪些 KV 要搬到 committed target KV。
- 拒绝后哪些临时 KV 要释放或覆盖。

这就是为什么 speculative 代码里会有 cache loc kernel、future map、verify buffer、draft/target attention backend 之间的同步。


In [ ]:
for path in [
    "python/sglang/srt/speculative/triton_ops/cache_locs.py",
    "python/sglang/srt/speculative/triton_ops/eagle.py",
    "python/sglang/srt/speculative/eagle_utils.py",
]:
    print("\n==", path)
    matches = [line for line in read_rel(path).splitlines() if "cache" in line.lower() or "accept" in line.lower()]
    for line in matches[:20]:
        print(line[:140])


## 小测试：算法名解析和插件保留名

这个测试只跑 Python enum/registry，不需要模型。


In [ ]:
from sglang.srt.speculative.spec_info import SpeculativeAlgorithm

for name in [None, "EAGLE", "eagle3", "standalone", "ngram", "none"]:
    algo = SpeculativeAlgorithm.from_string(name)
    print(name, "->", algo, "is_some=", algo.is_some(), "need_topk=", algo.need_topk() if hasattr(algo, "need_topk") else None)

try:
    SpeculativeAlgorithm.from_string("not-a-real-algo")
except ValueError as e:
    print("unknown name:", e)


## 贡献者注意点

- 新算法如果要进入主干，先明确它是否需要 draft KV、是否支持 overlap、是否需要 target hidden states。
- 所有 accept length 都会影响 scheduler 的输出 token 数、KV 提交数、metrics 和 finish 判断。
- 调试 spec decoding 时，先用 batch size 1、temperature 0、短 prompt 验证 token 序列，再放大到 overlap/batching。
